# Create a model to recommend movie to User based on user preference

In [2]:
## Import the necessary libraries
import pandas as pd 
import numpy as np 

In [3]:
df = pd.read_csv("data/movies_test.csv")
df.head()

,id,imdb_id,title,original_title,tagline,overview,genres,keywords,cast,director,...,vote_count,poster_path,backdrop_path,homepage,adult,belongs_to_collection,trailer_keys,watch_stream,watch_rent,watch_buy
0,1368337,tt33764258,The Odyssey,The Odyssey,Defy the gods.,"Odysseus, the legendary King of Ithaca, embark...",Adventure|Action|Fantasy,ship|trojan war|greek mythology|historical fic...,Matt Damon|Tom Holland|Anne Hathaway|Robert Pa...,Christopher Nolan,...,1057,/5rhTDKUhPYvpdQIijFIs5VoWsON.jpg,/tYuC9kUwqhpDQ3pv1kLMqyMF1Jw.jpg,https://www.universalpictures.com/movies/the-o...,False,NaN,AyIZ9tiiN8I|f_bKjZeJBBI|Mzw2ttJD2qQ,NaN,NaN,NaN
1,1108427,tt27419466,Moana,Moana,The ocean chose her for a reason.,"Teenage Moana answers the Ocean's call and, fo...",Family|Fantasy|Comedy|Adventure,sailboat|sea|friendship|ocean|leaving one's fa...,Catherine Lagaʻaia|Dwayne Johnson|Rena Owen|Jo...,Thomas Kail,...,135,/zKVgiv5qHCvCLT4A2ymJi5QeXDH.jpg,/mMkJq4dkQwfDieB9wRC9yPxDWv9.jpg,https://movies.disney.com/moana-2026,False,NaN,EEz5xbzYPKI|n7f6hlKsxxo,NaN,NaN,NaN
2,1275779,tt15047880,Disclosure Day,Disclosure Day,We deserve to know.,A cybersecurity expert becomes a whistleblower...,Science Fiction|Thriller,new york city|childhood trauma|alien|on the ru...,Emily Blunt|Josh O'Connor|Colin Firth|Colman D...,Steven Spielberg,...,1285,/AnJ8IQJI23hNpYXVNaythu061Ru.jpg,/flxau5Iu7bChQHsESqvGZ3FQRaI.jpg,https://www.disclosuredaymovie.com,False,NaN,icDuEHSxE-w|SCYT8vb2siQ,NaN,Amazon Video|Apple TV Store|Google Play Movies...,Amazon Video|Apple TV Store|Google Play Movies...
3,1339713,tt37287335,Obsession,Obsession,Be careful who you wish for…,"After breaking the mysterious ""One Wish Willow...",Horror|Thriller,obsession|supernatural|possession|wish|unrequi...,Michael Johnston|Inde Navarrette|Cooper Tomlin...,Curry Barker,...,3516,/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg,/r013C8Me2bZ0pUi0OWJRh0h7MzT.jpg,http://www.focusfeatures.com/obsession,False,NaN,gMC8kkwbIQQ,Peacock Premium|Peacock Premium Plus,Amazon Video|Apple TV Store|Google Play Movies...,Amazon Video|Apple TV Store|Google Play Movies...
4,1083381,tt26657236,Backrooms,Backrooms,See how far it goes.,A strange doorway appears in the basement of a...,Horror|Mystery|Science Fiction,california|monster|therapist|surrealism|childh...,Chiwetel Ejiofor|Renate Reinsve|Finn Bennett|L...,Kane Parsons,...,1951,/rhGx6E3qRNMgj3i5su2oukNHwIQ.jpg,/2aA8iPlMCks3k131ymFBAY6FL0b.jpg,https://app.a24films.com/release/backrooms/522,False,NaN,0HjdiohVOik,NaN,Amazon Video|Apple TV Store|Fandango At Home|Plex,Amazon Video|Apple TV Store|Fandango At Home


In [4]:
df.info()



<class 'pandas.DataFrame'>
RangeIndex: 546 entries, 0 to 545
Data columns (total 32 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     546 non-null    int64  
 1   imdb_id                543 non-null    str    
 2   title                  546 non-null    str    
 3   original_title         546 non-null    str    
 4   tagline                401 non-null    str    
 5   overview               544 non-null    str    
 6   genres                 543 non-null    str    
 7   keywords               493 non-null    str    
 8   cast                   543 non-null    str    
 9   director               545 non-null    str    
 10  release_date           546 non-null    str    
 11  status                 546 non-null    str    
 12  runtime                546 non-null    int64  
 13  original_language      546 non-null    str    
 14  spoken_languages       544 non-null    str    
 15  origin_country   

In [5]:
df["overview"] = df["overview"].fillna("")
df["genres"] = df["genres"].fillna("")
df["keywords"] = df["keywords"].fillna("")
df["tagline"] = df["tagline"].fillna("")

df["combined_text"] = (
    df["overview"] + " " +
    df["genres"].str.replace("|", " ", regex=False) + " " +
    df["keywords"].str.replace("|", " ", regex=False) + " " +
    df["tagline"]
)

print(df["combined_text"].iloc[0])

Odysseus, the legendary King of Ithaca, embarks on a long and perilous journey home following the Trojan War. Throughout his voyage, he is forced to confront the whims of gods, mythological monsters, and trials that stretch both his cunning and his humanity to the breaking point. Adventure Action Fantasy ship trojan war greek mythology historical fiction ancient world period drama based on song, poem or rhyme greek history voyage peplum ancient trojan horse mediterranean sea bronze age mythical odyssey sword and sandal epic fantasy Defy the gods.


In [6]:
print(repr(df["combined_text"].iloc[0]))

'Odysseus, the legendary King of Ithaca, embarks on a long and perilous journey home following the Trojan War. Throughout his voyage, he is forced to confront the whims of gods, mythological monsters, and trials that stretch both his cunning and his humanity to the breaking point. Adventure Action Fantasy ship trojan war greek mythology historical fiction ancient world period drama based on song, poem or rhyme greek history voyage peplum ancient trojan horse mediterranean sea bronze age mythical odyssey sword and sandal epic fantasy Defy the gods.'


In [7]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(df["combined_text"].tolist(), show_progress_bar=True)

print(embeddings.shape)



Batches: 100%|██████████| 18/18 [00:11<00:00,  1.54it/s]

(546, 384)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

# pick a movie by its row position (index 0 = The Odyssey, based on what you showed earlier)
target_idx = 0
target_vector = embeddings[target_idx].reshape(1, -1)

similarities = cosine_similarity(target_vector, embeddings)[0]

# get indices of the top 5 most similar (excluding itself)
top_indices = similarities.argsort()[::-1][1:6]

for i in top_indices:
    print(df.iloc[i]["title"], "-", similarities[i])

The Return - 0.87024343
Troy - 0.5880267
Dune: Part Two - 0.36538854
How to Train Your Dragon - 0.34881878
The Hobbit: The Battle of the Five Armies - 0.3439926
